# Compare power scans

Overlay the model curves of several pump-power scans to read off the effect of a configuration
change (P1 vs no-P1, filter A vs filter B, ...): $g^{(2)}(0)$ vs power, the inferred pump excess
$\sigma^2=g^{(2)}_0-1$, the intensity scaling $\langle I_n\rangle$ and the nonlinearity $K(n)$.

Each scan is one acquisition directory (its `MERGED/` folder holds one merged pickle per power).
Everything is saved under `results/<date>/<TITLE>/compare/`. Plots use the interactive **ipympl**
backend (drag to zoom).

In [ ]:
# ============================== EDIT ME ==============================
TITLE = "Jun16_P1_vs_noP1"

# {label: power-scan directory}.  A label is whatever varies between the scans.
SCANS = {
    "P1":    "data/Jun16/PowerScan_GaAs100_P1",
    "no P1": "data/Jun16/PowerScan_GaAs100_noP1",
}

HARMONICS = (3, 5)
TAU_IN_NS = 4.0
G2_METHOD = "delay"      # 'delay' | 'direct' | 'heralded'

# optional dense intensity scan per scan, for a smooth K(n): {label: dir}
DENSE = {}               # e.g. {"P1": "data/Jun16/PowerScan_GaAs100_P1_intensity"}
MALUS = {"p_max": 100.0, "theta0": 0.0, "offset": 0.0}
# ====================================================================

In [ ]:
%matplotlib widget
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "core.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
RESULTS_DIR = ROOT / "results"
print("Project root:", ROOT)

## 1. Build one analyzer per scan

In [ ]:
from src.report import discover_power_scan, discover_runs, PowerScanComparisonReport
from src.powerscan import PowerScanAnalyzer

analyzers = {}
for label, rel in SCANS.items():
    runs = discover_power_scan(ROOT / rel)
    dense = discover_runs(ROOT / DENSE[label]) if label in DENSE else None
    analyzers[label] = PowerScanAnalyzer(
        runs, harmonics=HARMONICS, tau_in_ns=TAU_IN_NS, g2_method=G2_METHOD,
        intensity_runs=dense, malus=(MALUS if dense else None))
    print(f"  {label:10s}: {len(runs)} powers "
          f"({analyzers[label].I0.min():g}-{analyzers[label].I0.max():g} mW)"
          f"{' + dense K(n)' if analyzers[label].has_dense else ''}")

## 2. Overlay + save

Saves every comparison figure under `results/<date>/<TITLE>/compare/`.

In [ ]:
rep = PowerScanComparisonReport(analyzers, title=TITLE, results_root=RESULTS_DIR,
                                harmonics=HARMONICS)
rep.plots(harmonics=HARMONICS, g2_ylim=(0.9, 1.6), sigma2_ylim=None,
          overview=True, show=True)

## 3. Quick zoom

Drag on any figure above, or re-render one comparison with explicit limits.

In [ ]:
fig, ax = rep.comp.plot_inferred_sigma2(harmonics=HARMONICS, ylim=(0, 0.03))
rep._display(fig)